In [1]:
#Importing the standard libraries
import numpy as np
np.set_printoptions(legacy='1.25',suppress=True)

import matplotlib.pyplot as plt
%matplotlib qt

#Importing the solver modules
from BaF_solver.system import System
from BaF_solver.obe_with_gradient import obe 
from BaF_solver.states import SigmaLevel,PiLevelParity
from BaF_solver.obe_with_gradient import Excitation, Static_Excitation
import time
import warnings
warnings.filterwarnings('ignore')
import datetime

/Users/mangesh/Documents/Git/BaF-solver/BaF_solver/obe_with_gradient.py:25: UserWarning: Could not detect diffeqpy. Only Python package for OBE solver available.
  warnings.warn(f"Could not detect diffeqpy. Only Python package for OBE solver available.")


In [21]:
import copy

#Reference ordering
Bz = 4600.0
Ez = 0.0
gs_order = [0,1]
es_order = ['1/2-','1/2+']
b=System(gs_order,es_order,B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],isotope = 138,ignore_mF = False)

start = time.perf_counter()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
b.sigma_Hamiltonian.Stark.generate_Stark()
b.pi_Hamiltonian.generate_bare()
b.pi_Hamiltonian.Zeeman.generate_Zeeman()
b.pi_Hamiltonian.Stark.generate_Stark()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()
b.pi_Hamiltonian.diagonalize()


G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,5)
E_global =b.pi_Hamiltonian.diagonalized_states
EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,5)

print(f"Bz = {Bz}.")
print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

G = copy.copy(G_global)#[3:4+1]
GH = copy.copy(GH_global)#[3:4+1,3:4+1]
E = copy.copy(E_global)
EH = copy.copy(EH_global)

vec_prev_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
vec_prev_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

sign_organizer_sigma_prev = np.ones(len(G))
sign_organizer_pi_prev = np.ones(len(E))

Bz = 4600.0.
Static Hamiltonian took : 0.031949250027537346s. 


In [22]:
GH_list = [] 
EH_list = [] 
H0_list = []
BR_list = []
Hint_1_list = []
Hstatic_int_list = []
from scipy.linalg import block_diag

Bz_list = np.arange(4600,4650,0.1)
for Bz in Bz_list:
    b=System(gs_order,es_order,B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],isotope = 138,ignore_mF = False)
    start = time.perf_counter()

    b.sigma_Hamiltonian.generate_bare()
    b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
    b.sigma_Hamiltonian.Stark.generate_Stark()
    b.pi_Hamiltonian.generate_bare()
    b.pi_Hamiltonian.Zeeman.generate_Zeeman()
    b.pi_Hamiltonian.Stark.generate_Stark()

    #Next diagonalize the Hamiltonian for this system
    b.sigma_Hamiltonian.diagonalize()
    b.pi_Hamiltonian.diagonalize()

    G_global  = b.sigma_Hamiltonian.diagonalized_states
    GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,5)
    E_global  = b.pi_Hamiltonian.diagonalized_states
    EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,5)


    G_temp = G_global#[3:4+1]
    GH_temp = GH_global#[3:4+1,3:4+1]
    
    E_temp = E_global#[0:16]
    EH_temp = EH_global#[0:16,0:16]
    

    vec_current_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
    vec_current_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

    #check the new ground state's overlap with the the previous ground states    permutation_sigma = []
    vec_current_hermit = vec_current_sigma.conj().T
    permutation_sigma = []



    NG = len(G_temp)
    NE = len(E_temp)
    for i in range(NG):
        #check current ith state ground state
        temp = vec_current_hermit@vec_prev_sigma[:,i]
        max_idx = np.argmax(np.abs(temp))
        permutation_sigma.append(max_idx)
        #ith state in te new list would be the max_idx state


    #check the new excited state's overlap with the the previous excited states
    permutation_pi = []
    sign_organizer_pi = np.zeros_like(sign_organizer_pi_prev)
    vec_current_hermit = vec_current_pi.conj().T
    for j in range(NE):
        #check current jth state excited state
        temp = vec_current_hermit@vec_prev_pi[:,j]
        max_idx = np.argmax(np.abs(temp))
        permutation_pi.append(max_idx)
    
    
    G_new = [G_temp[i] for i in permutation_sigma]    
    E_new = [E_temp[j] for j in permutation_pi]
    GH_new_diag = np.diag(GH_temp)[permutation_sigma]
    EH_new_diag = np.diag(EH_temp)[permutation_pi]
    GH_list.append(GH_new_diag)
    EH_list.append(EH_new_diag)

    H0 = np.diag(np.append(GH_new_diag,EH_new_diag))
    assert np.allclose(np.imag(H0),np.zeros(H0.shape))
    H0_list.append(H0.astype(np.float64))

    b.generate_branching_ratios(G_new,E_new)
    BR = b.branching_ratios
    #BR = np.nan_to_num(BR)
    assert np.allclose(np.imag(BR), np.zeros(BR.shape))

    BR_list.append(BR.astype(np.float64))

    start  = time.perf_counter()
    b.generate_interaction_Hamiltonian(G_new,E_new)
    Hint_1 = b.interaction_Hamiltonian
    

    #Read the operator to express the static electric field in the diagonalized basis
    U_sim_sigma = b.sigma_Hamiltonian.diagonalizing_Matrix
    Hstatic_int_sigma_temp = b.sigma_Hamiltonian.Stark.Z
    Hstatic_int_sigma_temp = U_sim_sigma.conj().T@Hstatic_int_sigma_temp@U_sim_sigma

    #Reorder with permutations
    H_temp_row = np.zeros_like(Hstatic_int_sigma_temp)
    H_temp_col = np.zeros_like(Hstatic_int_sigma_temp)
    for i in range(NG):
        H_temp_row[i,:] = Hstatic_int_sigma_temp[permutation_sigma[i],:]
    
    for i in range(NG):
        H_temp_col[:,i] = H_temp_row[:,permutation_sigma[i]]
    Hstatic_int_sigma = H_temp_col
    #Hstatic_int_sigma = Hstatic_int_sigma[3:4+1, 3:4+1]
    
    #Constructing the static electric field hamiltonian as an interaction rather than using it to diagonalizing the full hamiltonian
    U_sim_pi = b.pi_Hamiltonian.diagonalizing_Matrix
    Hstatic_int_pi_temp = b.pi_Hamiltonian.Stark.Z
    Hstatic_int_pi_temp = U_sim_pi.conj().T@Hstatic_int_pi_temp@U_sim_pi
    #Reorder with permutations
    H_temp_row = np.zeros_like(Hstatic_int_pi_temp)
    H_temp_col = np.zeros_like(Hstatic_int_pi_temp)
    for i in range(NE):
        H_temp_row[i,:] = Hstatic_int_pi_temp[permutation_pi[i],:]
    
    for i in range(NE):
        H_temp_col[:,i] = H_temp_row[:,permutation_pi[i]]
    Hstatic_int_pi = H_temp_col

    
    Hstatic_int = block_diag(Hstatic_int_sigma, Hstatic_int_pi)

    Hint_1_list.append(Hint_1)
    Hstatic_int_list.append(Hstatic_int)

    #rearragne the roder of the the vectors recorded as matrices too
    vec_prev_sigma,vec_prev_pi = vec_current_sigma[:,permutation_sigma],vec_current_pi[:,permutation_pi]
    

In [23]:
import scipy
from scipy.interpolate import interp1d,CubicSpline

# Given data
B_vals = np.arange(4600,4650,0.1)  
H0_vals = np.array(H0_list)             

BR_vals = np.array(BR_list)

Hint_1_vals = np.array(Hint_1_list)
Hint1_max = np.amax(np.abs(Hint_1_vals),axis=0)

Hstatic_int_vals = np.array(Hstatic_int_list)
Hstatic_int_max = np.amax(np.abs(Hstatic_int_vals),axis=0)

from BaF_solver.numba_cubicspline import numba_interpolate,get_interp_array


# Flatten for fast interpolation (shape: (10, 112*112))
H0_flat          = H0_vals.reshape(len(B_vals), -1)
BR_flat          = BR_vals.reshape(len(B_vals), -1)
Hint_1_flat      = Hint_1_vals.reshape(len(B_vals), -1)

Hstatic_int_flat = Hstatic_int_vals.reshape(len(B_vals), -1)


H0          = numba_interpolate(B_vals, H0_flat)
BR          = numba_interpolate(B_vals, BR_flat)
Hint_1      = numba_interpolate(B_vals, Hint_1_flat,real_imag = True)
Hstatic_int = numba_interpolate(B_vals, Hstatic_int_flat)


def get_interp_array(A_interp,shape,t,real_imag = False):
    if real_imag:
        real, imag = A_interp
        return (real(t).reshape(shape),imag(t).reshape(shape))
    else:
        return A_interp(t).reshape(shape)


n=len(E)+len(G)
plt.title("Without even parity")
tsigma = 8.192/4
for B in 4604.7433+np.arange(-0.01,0.01,0.001): #+np.array([0.004]):#
    Hint_real,Hint_imag = get_interp_array(Hint_1,(n,n),B,real_imag = True)
    plt.clf()
    #plt.subplot(121)
    plt.imshow(Hint_real,cmap='viridis')
    print(B,Hint_real[3,38])
    #plt.subplot(122)
    #plt.imshow(Hint_imag,cmap='viridis')
    #print(np.abs(my_obe_1.Hint[0][0](T)[7,2])**2 +np.abs(my_obe_1.Hint[0][1](T)[7,2])**2)
    
    plt.pause(0.1)

In [5]:
G[3],E[0]

((1+0j) |G = 0.5, N = 0, F1 = 0.5, F = 1.0, mF = 1.0>,
 (0.71057+0j) |J = 0.5-, F1 = 0.5, F = 0.0, mF = -0.0> + 
 (-0.70362+0j) |J = 0.5-, F1 = 0.5, F = 1.0, mF = 0.0>)

### SIngle run

In [6]:
### Single run
B_offset = -0.001
B_offset_range=np.arange(-0.015,0.015,0.0005)
r3_list = []

for B_offset in B_offset_range:
    v_beam = 616e-4 #cm/us
    V0 = 1.5 #Volts
    R = 1*1.26 #cm
    sigma_z = 0.479*R #cm
    sigma_t = sigma_z/v_beam
    pol = 0
    pos = 0.0
    ##
    E0 = 1.348*V0/R; dia = sigma_t*4 #in V/cm
    static_field = Static_Excitation(E0, pol, position = pos, diameter = dia, shape = "Unipolar")

    Gamma = 2.7 #Gamma = 2*np.pi*2.7
    tsigma = 8.192/4
    rabi = 1.0 * Gamma
    pol = 0
    groundState = G[3]
    excitedState = E[2]
    det = 0
    pos = -4/v_beam
    dia = 4*tsigma
    temp_field_L1_depletion = Excitation(rabi, pol, groundState, excitedState, detuning = det, position =  pos, diameter = dia, shape = "Uniform")
    temp_field_L2_depletion = Excitation(0 ,   pol, groundState, excitedState, detuning = det, position = -pos, diameter = dia, shape = "Uniform")

    optical_fields = [temp_field_L1_depletion,temp_field_L2_depletion]

    n=len(E)+len(G)
    #B_offset = -0.001
    Bactual = 4604.74325 + B_offset
    print(Bactual)
    H0_val_diag = np.diag(get_interp_array(H0,(n,n),Bactual,real_imag = False))
    print(H0_val_diag[3])
    BRs = get_interp_array(BR,(len(G),len(E)),Bactual,real_imag = False)
    #print(BRs[3,2])

    steps=500
    
    r_init = np.zeros((n,n),dtype = np.complex128)
    #r_init[15,15] = 1.0+0.0j

    for i in range(len(G)): #Initializing the density matrix considering a rot temperature of 4 K
        if i<4:
            r_init[i,i] = 1.0#/len(G)
        else:
            r_init[i,i] = 1.0#/len(G)

    r_diag = np.round(np.diag(r_init),3)
    for indx in [0,1,2,3,4,5,6,7]:
        print(f"r{indx} = {r_diag[indx].real}",end = ",")
    print("")


    test_factor = 30
    package = 'Python'
    obe_mode = 'symengine'
    overall_envelope = None
    #print('Creating obes')

    start = time.perf_counter()
    my_obe_1 = obe(optical_fields,[G,E],H0,Hint_1,BR,test_factor, mode = obe_mode,B_field = (Bactual,0.0),
                   E_stat_field = static_field, Hstatic_int = Hstatic_int,max_Hints = [Hint1_max],max_Hstatic = Hstatic_int_max,overall_envelope = overall_envelope)


    method = 'RK45'
    start = time.perf_counter()
    ans = my_obe_1.solve(steps,r_init,
                        max_step_size = 1/Gamma,#max_step_size,
                        package = package,
                        method = method)
    #print(f"Z Solve took {time.perf_counter() - start :.3f}.")
    rho = np.array(ans[-1]) #gives the solution at the end of the time
    r_init = rho.reshape(n,n)
    #print(f"Z solve took {time.time()-start} s.")
    r_diag = np.round(np.diag(r_init),3)
    #print(f"Trace : {np.sum(r_diag[4:24])}")
    for indx in [0,1,2,3,4,5,6,7]:
        print(f"r{indx} = {r_diag[indx].real}",end = ",")
    print("")

    r3_list.append(r_diag[3].real)
    """
    plt.clf()
    t = np.linspace(-4,1*4,ans.shape[0])*1/v_beam #in us

    for i in range(3,4):
        y = []
        for count in range(ans.shape[0]):
            A= ans[count].reshape(n,n)
            pops = np.diag(A).real
            y.append(pops[i])
        plt.plot(t,y,label = str(i))
        plt.ylim([0,1.25])
    plt.title(f"With even parity, B_field = {Bactual}")
    plt.legend();
    plt.pause(0.01)
    """

4604.72825
6445.812320803563
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.072,r2 = 0.928,r3 = 0.001,r4 = 0.999,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.72875
6445.813021104903
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.072,r2 = 0.928,r3 = 0.001,r4 = 0.999,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.72925
6445.813717809702
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.072,r2 = 0.928,r3 = 0.002,r4 = 0.998,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.72975
6445.814412190282
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.072,r2 = 0.928,r3 = 0.002,r4 = 0.998,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.7302500000005
6445.815108895082
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.072,r2 = 0.928,r3 = 0.003,r4 = 0.997,r5 = 1.0,r6 = 1.001,r7 = 1.0,
4604.730750000001
6445.8158091964215
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4

In [7]:
here

NameError: name 'here' is not defined

In [8]:
plt.plot(2*1.399e3*B_offset_range,r3_list)

In [ ]:
4604.7455-2*0.00115+3.36e-5+2.52e-5 + 0.001004

4604.7442628

In [ ]:
t = np.linspace(-4,1*4,ans.shape[0])*1/v_beam #in us

for i in range(3,4):
    y = []
    for count in range(ans.shape[0]):
        A= ans[count].reshape(n,n)
        pops = np.diag(A).real
        y.append(pops[i])
    plt.plot(t,y,label = str(i))
plt.title(f"With even parity, E_field = {b.E_stat_field[2]}")
plt.legend();
    

In [ ]:
G[23]

(0.07181-0j) |G = 0.5, N = 2, F1 = 1.5, F = 2.0, mF = -2.0> + 
(0.84676-0j) |G = 0.5, N = 2, F1 = 2.5, F = 2.0, mF = -2.0> + 
(-0.52711-0j) |G = 0.5, N = 2, F1 = 2.5, F = 3.0, mF = -2.0>

In [ ]:
err

NameError: name 'err' is not defined

In [ ]:
### Single run with rect pulse
B_offset = -0.001
B_offset_range=np.arange(-0.007,0.007,0.0001)
r3_list = []

for B_offset in B_offset_range:
    v_beam = 616e-4 #cm/us
    V0 = 0.75 #Volts
    R = 5*1.40 #cm
    sigma_z = 0.25*R #cm
    sigma_t = sigma_z/v_beam
    pol = 0
    pos = 0.0
    ##
    E0 = 2*V0/5/R; dia = sigma_t*4 #in V/cm
    static_field = Static_Excitation(E0, pol, position = pos, diameter = dia, shape = "Pulse")

    Gamma = 2.7 #Gamma = 2*np.pi*2.7
    tsigma = 8.192/4
    rabi = 1.0 * Gamma
    pol = 0
    groundState = G[3]
    excitedState = E[2]
    det = 0
    pos = -4/v_beam
    dia = 4*tsigma
    temp_field_L1_depletion = Excitation(rabi, pol, groundState, excitedState, detuning = det, position =  pos, diameter = dia, shape = "Uniform")
    temp_field_L2_depletion = Excitation(0 ,   pol, groundState, excitedState, detuning = det, position = -pos, diameter = dia, shape = "Uniform")

    optical_fields = [temp_field_L1_depletion,temp_field_L2_depletion]

    n=len(E)+len(G)
    #B_offset = -0.001
    Bactual = 4628.2439 + B_offset
    print(Bactual)
    H0_val_diag = np.diag(get_interp_array(H0,(n,n),Bactual,real_imag = False))
    print(H0_val_diag[3])
    BRs = get_interp_array(BR,(len(G),len(E)),Bactual,real_imag = False)
    #print(BRs[3,2])

    steps=500
    
    r_init = np.zeros((n,n),dtype = np.complex128)
    #r_init[15,15] = 1.0+0.0j

    for i in range(len(G)): #Initializing the density matrix considering a rot temperature of 4 K
        if i<4:
            r_init[i,i] = 1.0#/len(G)
        else:
            r_init[i,i] = 1.0#/len(G)

    r_diag = np.round(np.diag(r_init),3)
    for indx in [0,1,2,3,4,5,6,7]:
        print(f"r{indx} = {r_diag[indx].real}",end = ",")
    print("")


    test_factor = 30
    package = 'Python'
    obe_mode = 'symengine'
    overall_envelope = None
    #print('Creating obes')

    start = time.perf_counter()
    my_obe_1 = obe(optical_fields,[G,E],H0,Hint_1,BR,test_factor, mode = obe_mode,B_field = (Bactual,0.0),
                   E_stat_field = static_field, Hstatic_int = Hstatic_int,max_Hints = [Hint1_max],max_Hstatic = Hstatic_int_max,overall_envelope = overall_envelope)


    method = 'RK45'
    start = time.perf_counter()
    ans = my_obe_1.solve(steps,r_init,
                        max_step_size = 1/Gamma,#max_step_size,
                        package = package,
                        method = method)
    #print(f"Z Solve took {time.perf_counter() - start :.3f}.")
    rho = np.array(ans[-1]) #gives the solution at the end of the time
    r_init = rho.reshape(n,n)
    #print(f"Z solve took {time.time()-start} s.")
    r_diag = np.round(np.diag(r_init),5)
    #print(f"Trace : {np.sum(r_diag[4:24])}")
    for indx in [0,1,2,3,4,5,6,7]:
        print(f"r{indx} = {r_diag[indx].real}",end = ",")
    print("")

    r3_list.append(r_diag[3].real)
 



4604.736250000001
6445.823490803563
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.07171,r2 = 0.92829,r3 = 0.00011,r4 = 0.99989,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.73635
6445.823631048115
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.07171,r2 = 0.92829,r3 = 9e-05,r4 = 0.99991,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.73645
6445.8237712198115
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.07171,r2 = 0.92829,r3 = 6e-05,r4 = 0.99994,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.7365500000005
6445.823911299365
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.07171,r2 = 0.92829,r3 = 4e-05,r4 = 0.99996,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.736650000001
6445.824051267491
r0 = 1.0,r1 = 1.0,r2 = 1.0,r3 = 1.0,r4 = 1.0,r5 = 1.0,r6 = 1.0,r7 = 1.0,
r0 = 2.0,r1 = 1.07171,r2 = 0.92829,r3 = 3e-05,r4 = 0.99997,r5 = 1.0,r6 = 1.0,r7 = 1.0,
4604.73675
6445.8241911049
r0 

In [20]:
plt.plot(2*1.399e3*B_offset_range,r3_list,'.')

In [ ]:
plt.title("With even parity")
plt.imshow(my_obe_1.H0(-4/v_beam).reshape(12,12)[0:8,0:8])

In [ ]:
my_obe_1.H0(-4/v_beam)[0:,0:].shape

In [ ]:
E[7]

In [ ]:
B_offset = -0.001
Ez= 0.0001
Bz = 4604.7433 + B_offset
b=System([0],['1/2-','1/2+'],B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],ignore_mF = False)
start = time.perf_counter()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
b.sigma_Hamiltonian.Stark.generate_Stark()
b.pi_Hamiltonian.generate_bare()
b.pi_Hamiltonian.Zeeman.generate_Zeeman()
b.pi_Hamiltonian.Stark.generate_Stark()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()
b.pi_Hamiltonian.diagonalize()

G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,5)
E_global =b.pi_Hamiltonian.diagonalized_states
EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,5)

print(f"Bz = {Bz}.")
print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

G_temp = G_global#[3:4+1]
GH_temp = GH_global#[3:4+1,3:4+1]
E_temp = E_global#[0:16]
EH_temp = EH_global#[0:16,0:16]


H0 = np.diag(np.append(GH_global,EH_global))

b.generate_branching_ratios(G_global,E_global)
BR = b.branching_ratios
b.generate_interaction_Hamiltonian(G_global,E_global)
Hint_1=b.interaction_Hamiltonian


#Read the operator to express the static electric field in the diagonalized basis
U_sim_sigma = b.sigma_Hamiltonian.diagonalizing_Matrix
Hstatic_int_sigma_temp = b.sigma_Hamiltonian.Stark.Z
Hstatic_int_sigma_temp = U_sim_sigma.conj().T@Hstatic_int_sigma_temp@U_sim_sigma


U_sim_pi = b.pi_Hamiltonian.diagonalizing_Matrix
Hstatic_int_pi_temp = b.pi_Hamiltonian.Stark.Z
Hstatic_int_pi_temp = U_sim_pi.conj().T@Hstatic_int_pi_temp@U_sim_pi


Hstatic_int = block_diag(Hstatic_int_sigma_temp, Hstatic_int_pi_temp)
plt.title("With even parity")
#plt.imshow(BR.real,cmap='viridis')

In [ ]:
plt.title("With even parity")
plt.subplot(121)
plt.imshow(my_obe_1.Hint[0][0](-4/v_beam)[0:,0:],cmap='viridis')
plt.subplot(122)
plt.imshow(my_obe_1.Hint[0][1](-4/v_beam)[0:,0:],cmap='viridis')

In [ ]:
BR

In [ ]:
A = [-1,0,1]
for i,item in enumerate(A):
    print(item)
A

In [ ]:
a= np.array([[1,2,3,4],[5,6,7,8],[-1,-2,-3,-4],[-5,-6,-7,-8]])
b= np.array([0,2,1,3])
print(a)
print(a.T[b].T)

In [ ]:
a= np.array([[1,2,3,4],[5,6,7,8],[-1,-2,-3,-4],[-5,-6,-7,-8]])
b= np.array([0,2,1,3])
print(a)
print(a[:,b])

In [ ]:
a = -0.28+1j*0.5
a/np.sign(a)

print(np.abs(a),np.abs(a/np.sign(a)))

0.5730619512757762 0.5730619512757762
